In [1]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.utils import *
from module.prompt import *
from module.custom_model import *
from module.base_model import *
from module.tools import *

start_langsmith("final_music")
from typing import TypedDict, Annotated, List, Literal, Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks

LangSmith 추적을 시작합니다.
[프로젝트명]
final_music


In [2]:
class SubState(TypedDict):
    question: Annotated[str, "user input question"]  # 사용자 질의 or requeustion 질의
    plan: Annotated[list[str], "get plan_node"]  # llm 생성한 작업 계획서
    messages: Annotated[list, add_messages]  # 작업 수행 후 얻은 데이터
    current_steps: Annotated[str, "current step"]  # 현재 단계에서 수행한 결과
    past_steps: Annotated[list, add_messages]  # 현재 단계에서 수행한 결과
    answer: Annotated[str, " output final answer"]  # 최종 답변 출력
    human_feedback: bool = False
    db_query: Annotated[str, "db_query"]  # 현재 단계에서 수행한 결과

In [ ]:
def get_prompt_relevant_query():
    prompt = """ 
    # System :
    You are a database expert
    Determines the association between user requests and query execution results
    Please answer yes or no by confirming that the answer is relevant to the user's request

    # User Request:
    {current_steps}
    # AI Answer Query:
    {query}

    # Important : 
    response  is only `yes` or `no`
    """

    return ChatPromptTemplate.from_template(prompt)


def get_prompt_query_gen() -> ChatPromptTemplate:
    prompt = """You are a SQL expert with a strong attention to detail.

    You can define SQL queries, analyze queries results and interpretate query results to response an answer.

    Read the messages bellow and identify the user question, table schemas, query statement and query result, or error if they exist.

    1. If there's not any query result that make sense to answer the question, create a syntactically correct SQLite query to answer the user question. DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

    2. If you create a query, response ONLY the query statement. For example, "SELECT id, name FROM pets;"

    3. If a query was already executed, but there was an error. Response with the same error message you found. For example: "Error: Pets table doesn't exist"

    4. If a query was already executed successfully interpretate the response and answer the questio following this pattern: Answer: <<question answer>>. For example: "Answer: There three cats registered as adopted"

    5. Please add a semicolon (;) at the end of your SQL query.

    # User Request : 
    {current_steps}

    # placeholder:
    {placeholder}
    """
    return ChatPromptTemplate.from_template(prompt)


def get_prompt_answer():
    prompt = """ 
    You are a DB expert
    Print your final answer based on the query you ran previously

    # select:
    Map the information obtained from the select execution so that the user can easily understand it

    # User Request : 
    {current_steps}
    
    # placeholder:
    {placeholder}

    """
    return ChatPromptTemplate.from_template(prompt)

In [4]:
@tool
def db_query_tool(query: str) -> str:
    """
    Run SQL queries against a database and return results
    Returns an error message if the query is incorrect
    If an error is returned, rewrite the query, check, and retry
    """
    # 쿼리 실행
    db = get_db()
    result = db.run_no_throw(query)

    # 오류: 결과가 없으면 오류 메시지 반환
    if "Error" in result:
        return f"Error: {result} \n\n . Please rewrite your query and try again."
    # 정상: 쿼리 실행 결과 반환
    elif not result:
        return "Success: value is None"
    else:
        return f"Success: {result}"

In [ ]:
def instruction_node(state: SubState):
    plan = state["plan"]
    text_plan = "\n".join(f"{idx+1}. {text}" for idx, text in enumerate(plan))
    task = plan[0]
    task_str = f"""For the following plan: \n\n {text_plan} \n\n You are tasked with executing [step 1. {task}]."""
    return SubState({"messages": task_str, "current_steps": task_str})


def get_table_list_node(state: SubState):
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(
        tool for tool in tools if tool.name == "sql_db_list_tables"
    )
    llm_get_schema = llm.bind_tools(
        [sql_db_list_tables], tool_choice="sql_db_list_tables"
    )
    return SubState({"messages": llm_get_schema.invoke("")})


def get_all_table_node(state: SubState):
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(
        tool for tool in tools if tool.name == "sql_db_list_tables"
    )
    return ToolNode([sql_db_list_tables])


def get_one_table_info_node(state: SubState):
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    llm_with_schema = llm.bind_tools([sql_db_schema], tool_choice="sql_db_schema")
    state_message = state["messages"][-3:]
    result = llm_with_schema.invoke(state_message)
    return SubState({"messages": result})


def get_one_table_schema_node(state: SubState):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    return ToolNode([sql_db_schema])


def get_query_gen_node(state: SubState):
    prompt = get_prompt_query_gen()
    llm = get_gpt()
    # query_gen_llm = prompt | llm.bind_tools(
    #     [db_query_tool], tool_choice="db_query_tool"
    # )
    query_gen_llm = prompt | llm
    history = state["messages"]
    current_steps = state["current_steps"]
    query_gen = query_gen_llm.invoke(
        {"placeholder": history, "current_steps": current_steps}
    )
    db_query = query_gen.content
    # db_query = query_gen.tool_calls[0]["args"]["query"]
    return SubState({"messages": query_gen, "db_query": db_query})


def check_query_relavant(state: SubState):
    prompt = get_prompt_relevant_query()
    llm = get_gpt()
    query_gen_llm = prompt | llm.with_structured_output(GradeQuery)
    current_steps = state["current_steps"]
    query = state["db_query"]
    query_gen = query_gen_llm.invoke({"current_steps": current_steps, "query": query})
    return SubState({"messages": query_gen.datasource})


def execute_query(state: SubState):
    query = state["db_query"]
    response = db_query_tool(query)
    return SubState({"messages": AIMessage(content=response)})

#  실패시 get_query_check_node -> execute_query 로직 추가
def get_query_check_node(state: SubState):
    prompt = get_prompt_query_check()
    llm = get_gpt().bind_tools([db_query_tool], tool_choice="db_query_tool")
    chain = prompt | llm
    history = state["messages"][-1].tool_calls[0]["args"]["query"]
    query_gen = chain.invoke({"placeholder": [history]})
    return SubState({"messages": [query_gen]})


def answer_node(state: SubState):
    prompt = get_prompt_answer()
    llm = get_gpt()
    query_gen_llm = prompt | llm
    history = state["messages"]
    current_steps = state["current_steps"]
    result = query_gen_llm.invoke(
        {"placeholder": history, "current_steps": current_steps}
    )
    return SubState(
        {
            "past_steps": [result],
            "messages": [result],
        }
    )

In [ ]:

def query_relevant(
    state: SubState,
) -> Literal["execute_query", "get_one_table_info_node"]:
    latest_messages: str = state["messages"][-1].content
    if latest_messages == "yes":
        return "execute_query"
    else:
        return "get_one_table_info_node"

In [ ]:
sub_state_graph = StateGraph(SubState)
sub_state_graph.add_node("instruction_node", instruction_node)
sub_state_graph.add_node("get_table_list_node", get_table_list_node)
sub_state_graph.add_node("get_all_table_node", get_all_table_node)
sub_state_graph.add_node("get_one_table_info_node", get_one_table_info_node)
sub_state_graph.add_node("get_one_table_schema_node", get_one_table_schema_node)
sub_state_graph.add_node("get_query_gen_node", get_query_gen_node)
sub_state_graph.add_node("check_query_relavant", check_query_relavant)

sub_state_graph.add_node("get_query_check_node", get_query_check_node)
sub_state_graph.add_node("execute_query", execute_query)
sub_state_graph.add_node("answer_node", answer_node)


sub_state_graph.add_edge(START, "instruction_node")
sub_state_graph.add_edge("instruction_node", "get_table_list_node")
sub_state_graph.add_edge("get_table_list_node", "get_all_table_node")
sub_state_graph.add_edge("get_all_table_node", "get_one_table_info_node")
sub_state_graph.add_edge("get_one_table_info_node", "get_one_table_schema_node")
sub_state_graph.add_edge("get_one_table_schema_node", "get_query_gen_node")
sub_state_graph.add_edge("get_query_gen_node", "check_query_relavant")
sub_state_graph.add_conditional_edges(
    source="check_query_relavant", path=query_relevant
)

sub_state_graph.add_edge("execute_query", "answer_node")
sub_state_graph.add_edge("answer_node", END)

sub_ck = get_check_pointer()
sub_graph = sub_state_graph.compile(checkpointer=sub_ck)

In [8]:
config = get_runnable_config(recursion_limit=20, thread_id=get_random_uuid())
# messages  = "favorite music table 아무정보없어?"
# inputs = {'question':'팝송 가장 많이 팔린곡 5곡을 '}

In [9]:
# messages = "Kara Nielsen이름으로 팝송 5곡 FavoriteMusic에 추가해줘"
# messages = "FavoriteMusic 전체 리스트 "
inputs = {
    "question": "응 진행해",
    "plan": [
        "사용자의 FavoriteMusic 목록을 가져옵니다.",
        "가져온 FavoriteMusic 목록에서 서비스에서 제공되는 곡을 필터링합니다.",
        "최종적으로 서비스에서 제공되는 FavoriteMusic 목록을 출력합니다.",
    ],
    "messages": [
        HumanMessage(
            content="안녕 난 Kara Nielsen 이야",
            additional_kwargs={},
            response_metadata={},
            id="66e0c19c-9c28-468a-b3cf-86e7ff2f0313",
        ),
        AIMessage(
            content="실행 계획 : 사용자의 FavoriteMusic 목록을 가져옵니다. \n 사용 도구 : db_agent",
            additional_kwargs={},
            response_metadata={},
            id="7b6e88cd-17d4-4446-b48a-521efed67d3e",
        ),
        AIMessage(
            content="Kara Nielsen님의 FavoriteMusic 목록을 가져와서 서비스에서 제공되는 곡을 필터링한 후 알려드리겠습니다. 이 작업을 위해 데이터베이스 에이전트를 사용해도 될까요?",
            additional_kwargs={},
            response_metadata={
                "prompt_feedback": {"block_reason": 0, "safety_ratings": []},
                "finish_reason": "STOP",
                "model_name": "gemini-2.5-flash-lite",
                "safety_ratings": [],
            },
            id="run--5f45ed91-bd5f-44c5-8dd2-e7bbc8a82bad-0",
            usage_metadata={
                "input_tokens": 834,
                "output_tokens": 39,
                "total_tokens": 873,
                "input_token_details": {"cache_read": 0},
            },
        ),
        HumanMessage(
            content="응 진행해",
            additional_kwargs={},
            response_metadata={},
            id="f16a7c75-9761-4527-828a-339cf9cb44f8",
        ),
    ],
    "past_steps": [],
    "answer": "Kara Nielsen님의 FavoriteMusic 목록을 가져와서 서비스에서 제공되는 곡을 필터링한 후 알려드리겠습니다. 이 작업을 위해 데이터베이스 에이전트를 사용해도 될까요?",
    "human_feedback": "request_node",
    "next_agent": "db_agent",
}

stream_graph(
    graph=sub_graph,
    config=config,
    inputs=inputs,
)


🔄 Node: get_table_list_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: get_all_table_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Album, Artist, Customer, Employee, FavoriteMusic, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
🔄 Node: get_one_table_info_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: get_one_table_schema_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

CREATE TABLE "FavoriteMusic" (
	"FavoriteMusicId" INTEGER, 
	"CustomerId" INTEGER NOT NULL, 
	"TrackId" INTEGER NOT NULL, 
	PRIMARY KEY ("FavoriteMusicId"), 
	FOREIGN KEY("CustomerId") REFERENCES "Customer" ("CustomerId"), 
	FOREIGN KEY("TrackId") REFERENCES "Track" ("TrackId")
)

/*
3 rows from FavoriteMusic table:
FavoriteMusicId	CustomerId	TrackId
1	9	3256
2	9	3268
3	9	325
*/
🔄 Node: get_query_gen_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
SELECT FavoriteMusicId, CustomerId, TrackId FROM FavoriteMusic WHERE CustomerId 

C:\Users\ansgy\AppData\Local\Temp\ipykernel_3616\2813589085.py:77: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = db_query_tool(query)



🔄 Node: answer_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Kara Nielsen님(고객 ID: 9)의 FavoriteMusic 목록을 성공적으로 조회했습니다.  
현재 데이터베이스에 저장된 Kara Nielsen님의 즐겨찾기 음악 정보는 다음과 같습니다:

| FavoriteMusicId | CustomerId | TrackId |
|-----------------|------------|---------|
| 1               | 9          | 3256    |
| 2               | 9          | 3268    |
| 3               | 9          | 325     |
| 4               | 9          | 326     |
| 5               | 9          | 330     |

이제 다음 단계로, 이 목록에서 서비스에서 제공되는 곡만 필터링하는 작업을 진행할 수 있습니다.

In [10]:
snapshot = sub_graph.get_state(config)
snapshot.values["messages"]

[HumanMessage(content='안녕 난 Kara Nielsen 이야', additional_kwargs={}, response_metadata={}, id='66e0c19c-9c28-468a-b3cf-86e7ff2f0313'),
 AIMessage(content='실행 계획 : 사용자의 FavoriteMusic 목록을 가져옵니다. \n 사용 도구 : db_agent', additional_kwargs={}, response_metadata={}, id='7b6e88cd-17d4-4446-b48a-521efed67d3e'),
 AIMessage(content='Kara Nielsen님의 FavoriteMusic 목록을 가져와서 서비스에서 제공되는 곡을 필터링한 후 알려드리겠습니다. 이 작업을 위해 데이터베이스 에이전트를 사용해도 될까요?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--5f45ed91-bd5f-44c5-8dd2-e7bbc8a82bad-0', usage_metadata={'input_tokens': 834, 'output_tokens': 39, 'total_tokens': 873, 'input_token_details': {'cache_read': 0}}),
 HumanMessage(content='응 진행해', additional_kwargs={}, response_metadata={}, id='f16a7c75-9761-4527-828a-339cf9cb44f8'),
 HumanMessage(content='For the following plan: \n\n 1. 사용자의 FavoriteMusic 목록을 가져옵니다.\n2. 가져온 